# PT-Flow — CIFAR-10, full run (relay training across sessions)

Class-conditional CIFAR-10, all 10 classes, all 50,000 images, native 32×32 pixel space, conv
backbones. One-step (1-NFE) sampling. Designed to be run as a **relay**: each session trains for
~11 h, checkpoints, and the next session (possibly on a different account) picks up exactly where it
left off, with the ε-anneal state and a single continuous W&B curve.

## The budget, honestly

Per training sample this costs ~95 GFLOP (`K=32`, `eta_steps=2`, generator `ch=96`). A 2×T4 session
sustains roughly 6.5 TFLOPS in fp32, so:

| | |
|---|---|
| One 11 h session | ~2.7M samples = **~10,500 steps at batch 256** |
| Target: 800,000 steps × 256 | **205M samples** (EDM / iCT-class budget) |
| Sessions needed | **~76** |
| With 7 people relaying (~2.5 sessions/week each) | **~4–5 weeks** |

**Why batch 256 and not 1024.** Your 800,000-step figure is right — at batch **256**. 800k × 1024
would be 819M samples, ~8× DDPM's entire budget and ~93 days of continuous 2×T4 compute. At batch
256, 800k steps lands on 205M samples, which is the budget EDM and iCT actually use for CIFAR-10.
Smaller batches also give more optimizer steps per sample, which matters more than raw batch size at
a fixed FLOP budget.

**The single biggest speedup available** is fp16: T4 has fp16 tensor cores (65 TFLOPS vs 8.1 fp32),
and the codebase currently runs pure fp32 because Turing has no bf16. Adding fp16 AMP + `GradScaler`
would cut the 4–5 weeks to roughly **2 weeks**. It is not implemented yet.

> **Sidebar:** Internet **ON**, Accelerator **GPU T4 × 2**, secrets `GH_TOKEN` (GitHub read) and
> `WANDB_API_KEY`. To continue a run, also attach the previous session's output as a **Data source**.


## 1 — Setup, clone, W&B

For the relay to show up as **one** curve, every session must use the same `WANDB_RUN_ID` and the
same W&B project — and the 7 accounts must share one W&B API key (or one W&B team).


In [1]:
import os, sys, json, glob, time, shutil, subprocess

WORK      = "/kaggle/working"
REPO      = f"{WORK}/PT-FLow"
DATA_DIR  = f"{WORK}/ptflow_data/cifar10"
FID_NPZ   = f"{WORK}/cifar10_train_fid_stats.npz"
TORCH_HUB = f"{WORK}/torch_hub"
WORKDIR   = f"{WORK}/runs/ptflow_cifar10"
for d in (DATA_DIR, TORCH_HUB, os.path.dirname(WORKDIR)):
    os.makedirs(d, exist_ok=True)

# ===================== RUN-DEFINING KNOBS =====================
# These define the SCHEDULE and must be IDENTICAL in every relay session.
TARGET_STEPS = 800_000     # nominal schedule length (205M samples at B=256)
BATCH_SIZE   = 256         # global; 128/rank on 2 GPUs
K_PROPOSALS  = 32          # estimator budget (dominant cost term)
GEN_CH       = 96          # generator: 64 -> 7.1M, 96 -> 15.0M, 128 -> 25.8M
POT_CH       = 64          # potential: encoder->scalar, called K+1x per sample
EPS_FINAL    = 1e-3
EMA_DECAY    = 0.9999      # ~10k-step horizon, right for a long run
WANDB_PROJECT = "ptflow-cifar10"
WANDB_RUN_ID  = "cifar10-b256-k32-g96-v1"   # <-- same string in EVERY session
# ===================== PER-SESSION KNOBS ======================
SESSION_HOURS = 8.5        # training budget THIS session (Kaggle caps ~12h)
EVAL_HOURS    = 2.0        # reserved for the evaluation suite below
RUN_EVAL      = True       # set False on intermediate relay legs to train longer
FID_SWEEP_N   = 10_000     # samples per guidance weight in the sweep
FID_FINAL_N   = 50_000     # headline FID at the best w
GITHUB_REPO   = "github.com/kraihan/PT-FLow.git"
# =============================================================

if not os.path.isdir(REPO):
    from kaggle_secrets import UserSecretsClient
    try:
        _tok = UserSecretsClient().get_secret("GH_TOKEN")
    except Exception as e:
        raise RuntimeError("Add a Kaggle Secret GH_TOKEN with a GitHub read token.") from e
    r = subprocess.run(["git", "clone", f"https://{_tok}@{GITHUB_REPO}", REPO],
                       capture_output=True, text=True)
    del _tok
    if r.returncode != 0:
        raise RuntimeError("clone failed: " + r.stderr[:400])
    print("cloned")
else:
    print("repo already present")

get_ipython().system("pip install -q torch-fidelity einops absl-py wandb")

import torch
NGPU = max(1, torch.cuda.device_count())
print("torch", torch.__version__, "| GPUs", NGPU,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

# ---- W&B: one shared run across every relay session ----
USE_WANDB = True
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print(f"W&B enabled -> project '{WANDB_PROJECT}', run id '{WANDB_RUN_ID}'")
except Exception as e:
    USE_WANDB = False
    print("W&B disabled (no WANDB_API_KEY secret); metrics go to the local jsonl instead")

os.environ.update(
    CIFAR10_PATH=DATA_DIR, CIFAR10_FID_NPZ=FID_NPZ, TORCH_HUB_DIR=TORCH_HUB,
    PYTHONPATH=REPO, OMP_NUM_THREADS="2", TOKENIZERS_PARALLELISM="false",
)
if REPO not in sys.path:
    sys.path.insert(0, REPO)


cloned
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 6.2 MB/s eta 0:00:00
torch 2.10.0+cu128 | GPUs 2 | Tesla T4
W&B disabled (no WANDB_API_KEY secret); metrics go to the local jsonl instead


### 1b — Verify the clone

No source patching: this run uses the repo as published, on all 10 classes and all 50k images.


In [2]:
_need = ["ptflow/models/unet.py", "configs/cifar10_unet.yaml",
         "configs/cifar10_unet_smoke.yaml", "scripts/make_ref_stats.py"]
_missing = [f for f in _need if not os.path.exists(f"{REPO}/{f}")]
assert not _missing, f"missing from the clone: {_missing} -- push them, or delete {REPO} and re-clone"

_dit = open(f"{REPO}/ptflow/models/dit.py").read()
_dst = open(f"{REPO}/ptflow/utils/dist_util.py").read()
_trn = open(f"{REPO}/ptflow/train/trainer.py").read()
_bld = open(f"{REPO}/ptflow/models/builder.py").read()
checks = {
    "RoPE buffers not aliased": "copy=True" in _dit,
    "DDP broadcast_buffers disabled": ("broadcast_buffers=False" in _dst
                                        or '"broadcast_buffers": False' in _dst),
    "ESS all-reduced before control flow": "all_reduce_mean" in _dst and "all_reduce_mean" in _trn,
    "builder has kind:unet": "unet" in _bld,
}
for k, v in checks.items():
    print(f"  [{'OK ' if v else 'MISSING'}] {k}")
assert all(checks.values()), "clone is stale -- push the current repo and re-clone"
print("\nrepo is current")


  [OK ] RoPE buffers not aliased
  [OK ] DDP broadcast_buffers disabled
  [OK ] ESS all-reduced before control flow
  [OK ] builder has kind:unet

repo is current


## 2 — CIFAR-10 + the 50k FID reference

Standard protocol: score generated samples against Inception statistics of the **full 50,000-image
train split**, built with the repo's own TF-compatible Inception (the same one `inference.py
evaluate` uses — mixing extractors silently produces meaningless FID).

Cached in `/kaggle/working`, so on a relay leg with the previous output attached this is instant.


In [3]:
GDRIVE_ID = "1AMESa9etn7VmnXb3GPbO2DWYttULbRvY"
CIFAR_MD5 = "c58f30108f718f92721af3b95e74349a"

# reuse a reference carried in from a previous session if one is attached
if not os.path.exists(FID_NPZ):
    for cand in glob.glob("/kaggle/input/*/cifar10_train_fid_stats.npz"):
        shutil.copy(cand, FID_NPZ); print("reused FID reference from", cand); break

if not os.path.exists(f"{DATA_DIR}/cifar-10-batches-py/data_batch_1"):
    get_ipython().system("pip install -q gdown")
    get_ipython().system(f"gdown {GDRIVE_ID} -O {DATA_DIR}/cifar-10-python.tar.gz")
    import hashlib
    got = hashlib.md5(open(f"{DATA_DIR}/cifar-10-python.tar.gz", "rb").read()).hexdigest()
    assert got == CIFAR_MD5, f"gdown fetched the wrong file (md5 {got})"
    from torchvision.datasets import CIFAR10
    CIFAR10(root=DATA_DIR, train=True, download=True)
    CIFAR10(root=DATA_DIR, train=False, download=True)

if not os.path.exists(FID_NPZ):
    rc = os.system(f"cd {REPO} && python scripts/make_ref_stats.py --dataset cifar10 "
                   f"--out {FID_NPZ} --batch-size 250")
    assert rc == 0, "FID reference build failed"

import numpy as np
_z = np.load(FID_NPZ)
assert _z["mu"].shape == (2048,) and _z["sigma"].shape == (2048, 2048)
print("FID reference ready:", {k: _z[k].shape for k in _z.files})


Downloading...
From (original): https://drive.google.com/uc?id=1AMESa9etn7VmnXb3GPbO2DWYttULbRvY
From (redirected): https://drive.google.com/uc?id=1AMESa9etn7VmnXb3GPbO2DWYttULbRvY&confirm=t&uuid=092e1247-07e1-4f7e-a280-163e19195f81
To: /kaggle/working/ptflow_data/cifar10/cifar-10-python.tar.gz
100%|████████████████████████████████████████| 170M/170M [00:03<00:00, 48.3MB/s]


Creating feature extractor "inception-v3-compat" with features ['2048', 'logits_unbiased']
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /kaggle/working/torch_hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 471MB/s]


Extracting Inception features for 50000 images ...
Wrote /kaggle/working/cifar10_train_fid_stats.npz  (mu (2048,), sigma (2048, 2048))
FID reference ready: {'mu': (2048,), 'sigma': (2048, 2048)}


## 3 — Config

`anneal_steps` is pinned to `0.8 × TARGET_STEPS` and **never** to the per-session step count — that
is what keeps the ε schedule continuous across the relay. The LR schedule is `const`, so it is
session-independent by construction. Only `train.total_steps` varies per session (§5), acting as
"where this leg stops".


In [4]:
import yaml
from ptflow.utils.misc import load_config

cfg = json.loads(json.dumps(load_config(f"{REPO}/configs/cifar10_unet.yaml")))

cfg["logging"] = {"use_wandb": USE_WANDB, "log_every_k": 50,
                  "project": WANDB_PROJECT, "run_id": WANDB_RUN_ID}
cfg["dataset"]["num_classes"] = 10
cfg["dataset"]["batch_size"] = BATCH_SIZE
cfg["dataset"]["eval_batch_size"] = 250
cfg["dataset"]["kwargs"]["num_workers"] = 2
cfg["dataset"]["use_aug"] = True

cfg["model"]["generator"]["ch"] = GEN_CH
cfg["model"]["potential"]["ch"] = POT_CH

cfg["train"]["micro_batches"] = 4          # K*B_mb sequences must fit in 15GB
cfg["train"]["eta_steps"] = 2
cfg["train"]["ema_decay"] = EMA_DECAY
cfg["train"]["ema_warmup"] = 5000
cfg["train"]["diag_every"] = 5000          # prox_multistart = 80 sequential fwd+bwd
cfg["train"]["save_per_step"] = 2000       # survive a session kill
cfg["train"]["keep_last"] = 2
cfg["train"]["keep_every"] = 100000
cfg["train"]["eval_per_step"] = 10**9      # eval runs below, not inline
cfg["train"]["potential_loss"]["K"] = K_PROPOSALS
cfg["optimizer"]["warmup_steps"] = 5000
cfg["anneal"]["eps_final"] = EPS_FINAL
cfg["anneal"]["anneal_steps"] = int(0.8 * TARGET_STEPS)    # SCHEDULE-DEFINING, never per-session
cfg["eval"]["enabled"] = False

CONFIG = f"{REPO}/configs/cifar10_pro.yaml"

def write_config(stop_at_step, use_wandb=None):
    cfg["train"]["total_steps"] = int(stop_at_step)
    if use_wandb is not None:
        cfg["logging"]["use_wandb"] = bool(use_wandb)
    yaml.safe_dump(cfg, open(CONFIG, "w"), sort_keys=False)

write_config(TARGET_STEPS)

from ptflow.models.builder import build_networks
pot, gen, fs = build_networks(load_config(CONFIG))
n_p = sum(q.numel() for q in pot.parameters()); n_g = sum(q.numel() for q in gen.parameters())
print(f"potential {n_p/1e6:.2f}M | generator {n_g/1e6:.2f}M | d={fs.dim}")

from torch.utils.flop_counter import FlopCounterMode
_x = torch.randn(2, 32, 32, 3); _c = torch.zeros(2, dtype=torch.long)
_f = {}
for _n, _call in (("pot", lambda: pot(_x, _c)), ("gen", lambda: gen(_x, _c, 0.5))):
    _m = FlopCounterMode(display=False)
    with _m: _call()
    _f[_n] = _m.get_total_flops() / 2 / 1e9
PER_SAMPLE_GFLOP = (3*K_PROPOSALS + 3 + 2*4) * _f["pot"] + (1 + 2*3) * _f["gen"]
print(f"potential {_f['pot']:.2f} GFLOP/img | generator {_f['gen']:.2f} GFLOP/img")
print(f"=> {PER_SAMPLE_GFLOP:.0f} GFLOP per training sample "
      f"({100*(3*K_PROPOSALS+3+2*4)*_f['pot']/PER_SAMPLE_GFLOP:.0f}% potential)")
print(f"\nTARGET {TARGET_STEPS:,} steps x B={BATCH_SIZE} = "
      f"{TARGET_STEPS*BATCH_SIZE/1e6:.0f}M samples ({TARGET_STEPS*BATCH_SIZE/50000:.0f} epochs)")
del pot, gen


potential 1.66M | generator 14.99M | d=3072
potential 0.43 GFLOP/img | generator 7.02 GFLOP/img
=> 95 GFLOP per training sample (48% potential)

TARGET 800,000 steps x B=256 = 205M samples (4096 epochs)


## 4 — Sanity (20 steps, tiny model)

End-to-end gate on both GPUs before committing hours.


In [5]:
smoke = json.loads(json.dumps(load_config(f"{REPO}/configs/cifar10_unet_smoke.yaml")))
smoke["dataset"]["num_classes"] = 10
smoke["dataset"]["batch_size"] = 32
smoke.setdefault("eval", {})["enabled"] = False
smoke["logging"] = {"use_wandb": False, "log_every_k": 5}
yaml.safe_dump(smoke, open(f"{REPO}/configs/cifar10_pro_smoke.yaml", "w"), sort_keys=False)

get_ipython().system(f"rm -rf {WORK}/runs/smoke")
rc = os.system(f"cd {REPO} && torchrun --nproc_per_node={NGPU} --master_port=29551 "
               f"train.py --config configs/cifar10_pro_smoke.yaml --workdir {WORK}/runs/smoke")
assert rc == 0, "sanity run failed - read the traceback before spending GPU hours"
print(f"\nSANITY OK on {NGPU} GPU(s)")


  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Grad strides do not match bucket view strides. This may indicate grad was not created according to the gradient layout contract, or that the param's strides changed since DDP was constructed.  This is not an error, but may impair performance.
grad.sizes() = [32, 32, 1, 1], strides() = [32, 1, 32, 32]
bucket_view.sizes() = [32, 32, 1, 1], strides() = [32, 1, 1, 1] (Triggered internally at /pytorch/torch/csrc/distributed/c10d/reducer.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Grad strides do not match bucket view strides. This may indicate grad was not created according to the gradient layout contract, or that the param's strides changed since DDP was constructed.  This is not an error, but may impair performance.
grad.


SANITY OK on 2 GPU(s)


## 5 — Timing probe → size this session, and get the ETA to target

Runs the **real** model briefly and reads the per-step times the trainer logs. The probe writes to a
throwaway workdir with W&B off (the local `metrics.jsonl` only exists when W&B is disabled).


In [6]:
PROBE_STEPS = 60
probe_dir = f"{WORK}/runs/probe"
get_ipython().system(f"rm -rf {probe_dir}")

write_config(PROBE_STEPS, use_wandb=False)          # probe: local jsonl, no W&B pollution
rc = os.system(f"cd {REPO} && torchrun --nproc_per_node={NGPU} --master_port=29553 "
               f"train.py --config configs/cifar10_pro.yaml --workdir {probe_dir}")
assert rc == 0, "timing probe failed"

rows = [json.loads(l) for l in open(f"{probe_dir}/log/metrics.jsonl")]
times = [r["step_time"] for r in rows if "step_time" in r]
warm = times[len(times)//2:] or times
SEC_PER_IT = sorted(warm)[len(warm)//2]

# where does this session start from?
prev = sorted(glob.glob("/kaggle/input/*/runs/ptflow_cifar10/checkpoints/state_*.pt")
              + glob.glob(f"{WORKDIR}/checkpoints/state_*.pt"))
RESUME_STEP = int(os.path.basename(prev[-1]).split("_")[1].split(".")[0]) if prev else 0

session_steps = int(SESSION_HOURS * 3600 / SEC_PER_IT)
STOP_AT = min(TARGET_STEPS, RESUME_STEP + session_steps)
write_config(STOP_AT, use_wandb=USE_WANDB)

samples_s = BATCH_SIZE / SEC_PER_IT
print(f"measured {SEC_PER_IT:.3f} s/it at B={BATCH_SIZE} on {NGPU} GPU(s)"
      f"   = {samples_s:.0f} samples/s = {samples_s*3600/1e6:.2f}M samples/hour")
print(f"effective throughput: {samples_s*PER_SAMPLE_GFLOP/1e3:.1f} TFLOPS\n")

print(f"resume from step : {RESUME_STEP:,}")
print(f"this session     : +{STOP_AT-RESUME_STEP:,} steps -> stop at {STOP_AT:,}")
print(f"target           : {TARGET_STEPS:,}  ({100*RESUME_STEP/TARGET_STEPS:.1f}% done now, "
      f"{100*STOP_AT/TARGET_STEPS:.1f}% after this session)")

remaining = TARGET_STEPS - STOP_AT
if remaining > 0:
    sess_left = remaining / max(1, session_steps)
    print(f"\nremaining        : {remaining:,} steps = {sess_left:.0f} more sessions")
    print(f"  1 person  (~2.5 sessions/wk): {sess_left/2.5:.0f} weeks")
    print(f"  7 people  (~17 sessions/wk) : {sess_left/17:.0f} weeks")
    print(f"  with fp16 AMP (~2.5x)       : {sess_left/17/2.5:.1f} weeks with 7 people")
else:
    print("\nTARGET REACHED - this session finishes the schedule.")


  0%|          | 0/60 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Grad strides do not match bucket view strides. This may indicate grad was not created according to the gradient layout contract, or that the param's strides changed since DDP was constructed.  This is not an error, but may impair performance.
grad.sizes() = [192, 192, 1, 1], strides() = [192, 1, 192, 192]
bucket_view.sizes() = [192, 192, 1, 1], strides() = [192, 1, 1, 1] (Triggered internally at /pytorch/torch/csrc/distributed/c10d/reducer.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Grad strides do not match bucket view strides. This may indicate grad was not created according to the gradient layout contract, or that the param's strides changed since DDP was constructed.  This is not an error, but may impair performanc

measured 3.418 s/it at B=256 on 2 GPU(s)   = 75 samples/s = 0.27M samples/hour
effective throughput: 7.1 TFLOPS

resume from step : 0
this session     : +8,951 steps -> stop at 8,951
target           : 800,000  (0.0% done now, 1.1% after this session)

remaining        : 791,049 steps = 88 more sessions
  1 person  (~2.5 sessions/wk): 35 weeks
  7 people  (~17 sessions/wk) : 5 weeks
  with fp16 AMP (~2.5x)       : 2.1 weeks with 7 people


## 6 — Train this leg

Resumes from the newest checkpoint found in `/kaggle/input/**` or the workdir, restoring model, EMA,
optimizer **and anneal state**, and appends to the same W&B run.

**Relay handoff, for the next person:**
1. When this notebook finishes, **Save Version** so `/kaggle/working` is preserved.
2. Share that version's output as a dataset (public, or with the group).
3. The next person attaches it as a **Data source**, keeps `TARGET_STEPS`, `BATCH_SIZE`,
   `K_PROPOSALS`, `GEN_CH`, `WANDB_RUN_ID` **identical**, and runs from the top.

Two numbers certify the run: **`ess`** ≥ 0.3 (below that the trainer pauses the anneal and doubles
the generator:potential ratio on its own), and **`L_theta_phi_units`** converging. Raw `L_theta`
grows like 1/ε by design — that is not divergence.


In [ ]:
os.makedirs(f"{WORKDIR}/checkpoints", exist_ok=True)
inp = sorted(glob.glob("/kaggle/input/*/runs/ptflow_cifar10/checkpoints/state_*.pt"))
if inp and not glob.glob(f"{WORKDIR}/checkpoints/state_*.pt"):
    for p in inp[-2:]:
        shutil.copy(p, f"{WORKDIR}/checkpoints/{os.path.basename(p)}")
    print(f"resuming from {os.path.basename(inp[-1])}")
elif glob.glob(f"{WORKDIR}/checkpoints/state_*.pt"):
    print("checkpoints already in the workdir; trainer resumes from the latest")
else:
    print("fresh run (step 0)")

_t0 = time.time()
rc = os.system(f"cd {REPO} && torchrun --nproc_per_node={NGPU} --master_port=29550 "
               f"train.py --config configs/cifar10_pro.yaml --workdir {WORKDIR}")
_h = (time.time() - _t0) / 3600
print(f"\ntraining wall time: {_h:.2f} h")
if rc != 0:
    print("WARNING: trainer exited non-zero (session limit? OOM?). Checkpoints are on disk; "
          "evaluation below uses the newest.")

ck = sorted(glob.glob(f"{WORKDIR}/checkpoints/state_*.pt"))
DONE_STEP = int(os.path.basename(ck[-1]).split("_")[1].split(".")[0]) if ck else RESUME_STEP
print(f"progress: {DONE_STEP:,} / {TARGET_STEPS:,}  ({100*DONE_STEP/TARGET_STEPS:.1f}%)")
get_ipython().system(f"ls -lh {WORKDIR}/checkpoints/")


fresh run (step 0)


  0%|          | 0/8951 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Grad strides do not match bucket view strides. This may indicate grad was not created according to the gradient layout contract, or that the param's strides changed since DDP was constructed.  This is not an error, but may impair performance.
grad.sizes() = [192, 192, 1, 1], strides() = [192, 1, 192, 192]
bucket_view.sizes() = [192, 192, 1, 1], strides() = [192, 1, 1, 1] (Triggered internally at /pytorch/torch/csrc/distributed/c10d/reducer.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Grad strides do not match bucket view strides. This may indicate grad was not created according to the gradient layout contract, or that the param's strides changed since DDP was constructed.  This is not an error, but may impair performa

## 7 — Evaluation

Skipped on intermediate relay legs when `RUN_EVAL = False` — on a mid-relay session it is usually
better to spend the 2 hours training. Run it on the first session (to confirm the pipeline), then
every few legs to track progress, and always on the final one.


In [ ]:
from IPython.display import Image as IPImage, display

ckpts = sorted(glob.glob(f"{WORKDIR}/checkpoints/state_*.pt"))
assert ckpts, "no checkpoints - did training run?"
CKPT = ckpts[-1]
print("evaluating", os.path.basename(CKPT), "| RUN_EVAL =", RUN_EVAL)

CIFAR_NAMES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
if RUN_EVAL:
    for w in (0.0, 0.5):
        out = f"{WORK}/samples_w{w}.png"
        get_ipython().system(
            f'cd {REPO} && python inference.py sample --ckpt {CKPT} '
            f'--config configs/cifar10_pro.yaml --mode A --w {w} '
            f'--class-ids "0,1,2,3,4,5,6,7,8,9" --num-rows 8 --seed 42 --save-path {out}')
        print(f"Mode A (1-NFE), w={w}   columns: {', '.join(CIFAR_NAMES)}")
        display(IPImage(out))


### 7a — FID / IS: guidance sweep, then the headline number

Sweep `w` cheaply to find the best guidance weight, then re-run **only** the winner at 50k. A 50k
run per `w` would cost hours for no extra information.


In [ ]:
import pandas as pd

def run_eval(w, n, mode="A", extra=""):
    jout = f"{WORK}/eval_{mode}_w{w}_n{n}.json"
    rc = os.system(f"cd {REPO} && torchrun --nproc_per_node={NGPU} --master_port=29552 "
                   f"inference.py evaluate --ckpt {CKPT} --config configs/cifar10_pro.yaml "
                   f"--mode {mode} --w {w} --num-samples {n} --gen-bsz 250 {extra} "
                   f"--json-out {jout}")
    return json.load(open(jout)) if (rc == 0 and os.path.exists(jout)) else None

sweep, BEST_W, final = {}, 0.0, None
if RUN_EVAL:
    for w in (0.0, 0.25, 0.5, 1.0, 1.5):
        r = run_eval(w, FID_SWEEP_N)
        if r:
            sweep[w] = r
            print(f"  w={w:<5} FID {r.get('fid', float('nan')):8.2f}   IS {r.get('isc_mean', 0):6.2f}")
    assert sweep, "every sweep evaluation failed"
    BEST_W = min(sweep, key=lambda k: sweep[k].get("fid", float("inf")))
    print(f"\nbest w = {BEST_W} (FID {sweep[BEST_W]['fid']:.2f} at {FID_SWEEP_N} samples)")
    print(f"headline run: {FID_FINAL_N} samples ...")
    final = run_eval(BEST_W, FID_FINAL_N)
    if final:
        print(f"\n  FID {final['fid']:.2f}   IS {final.get('isc_mean',0):.2f}"
              f" +/- {final.get('isc_std',0):.2f}   (1-NFE, w={BEST_W}, {FID_FINAL_N} samples)")


### 7b — Mode B: the quality / compute dial

`n` damped fixed-point steps on the prox residual, starting from the Mode-A output. No retraining.
FID should fall monotonically with `n` — that curve is the evidence the generator really is
approximating the prox rather than an unrelated map.


In [ ]:
import matplotlib.pyplot as plt

mode_b = {}
if RUN_EVAL:
    for n in (0, 1, 2, 4):
        r = sweep.get(BEST_W) if n == 0 else run_eval(
            BEST_W, FID_SWEEP_N, mode="B", extra=f"--refine-steps {n} --gamma 0.5")
        if r:
            mode_b[n] = r.get("fid")
            print(f"  n={n}  NFE={n+1}  FID {mode_b[n]:8.2f}")
    if len(mode_b) > 1:
        ks = sorted(mode_b)
        plt.figure(figsize=(6, 4))
        plt.plot([k+1 for k in ks], [mode_b[k] for k in ks], "o-")
        plt.xlabel("NFE (1 = Mode A)"); plt.ylabel("FID"); plt.grid(alpha=0.3)
        plt.title(f"Mode-B refinement (w={BEST_W})"); plt.show()


### 7c — Exact normalized likelihood (w = 0)

The thing no other one-step model reports. Each value is an IWAE-style **bound**, so the honest
report is a ladder — NLL should decrease as `K_eval` grows, and flatten. Valid at `w = 0` only.


In [ ]:
if RUN_EVAL:
    LL = f"{WORK}/likelihood.json"
    rc = os.system(f"cd {REPO} && python inference.py likelihood --ckpt {CKPT} "
                   f"--config configs/cifar10_pro.yaml --num-samples 512 --batch-size 32 "
                   f"--ladder 16,64,256,1024 --json-out {LL}")
    if rc == 0 and os.path.exists(LL):
        ll = json.load(open(LL))
        ks = sorted(int(k.split("K")[1]) for k in ll if k.startswith("nll_K"))
        print(f"  {'K_eval':>8}  {'NLL (nats)':>12}  {'bits/dim':>10}")
        for k in ks:
            print(f"  {k:>8}  {ll[f'nll_K{k}']:12.2f}  {ll[f'nll_K{k}']/(3072*0.6931):10.4f}")
        plt.figure(figsize=(6, 4))
        plt.semilogx(ks, [ll[f"nll_K{k}"] for k in ks], "o-", base=2)
        plt.xlabel("K_eval"); plt.ylabel("NLL (nats)"); plt.grid(alpha=0.3)
        plt.title("IWAE ladder - should tighten monotonically"); plt.show()


## 8 — Session summary + handoff

In [ ]:
summary = {
    "run_id": WANDB_RUN_ID,
    "step": DONE_STEP, "target": TARGET_STEPS,
    "progress_pct": round(100*DONE_STEP/TARGET_STEPS, 2),
    "samples_seen_M": round(DONE_STEP*BATCH_SIZE/1e6, 1),
    "epochs": round(DONE_STEP*BATCH_SIZE/50000, 1),
    "batch_size": BATCH_SIZE, "K": K_PROPOSALS, "gen_ch": GEN_CH,
    "sec_per_it": round(SEC_PER_IT, 3),
}
if final:
    summary.update(FID_1NFE=round(final["fid"], 2),
                   IS=round(final.get("isc_mean", 0), 2), best_w=BEST_W)
if mode_b:
    summary["mode_b_fid_by_nfe"] = {k+1: (round(v, 2) if v else None) for k, v in mode_b.items()}
pd.DataFrame([summary]).T.to_csv(f"{WORK}/session_summary.csv", header=["value"])
print(json.dumps(summary, indent=2))

left = TARGET_STEPS - DONE_STEP
print(f"""
--------------------------------------------------------------------
HANDOFF -- next session

  1. Save Version (so /kaggle/working is preserved)
  2. Share that output as a dataset with the next person
  3. They attach it as a Data source and run this notebook unchanged

  Keep IDENTICAL: TARGET_STEPS={TARGET_STEPS}, BATCH_SIZE={BATCH_SIZE},
                  K_PROPOSALS={K_PROPOSALS}, GEN_CH={GEN_CH},
                  WANDB_RUN_ID='{WANDB_RUN_ID}'
  Change freely : SESSION_HOURS, RUN_EVAL

  progress {DONE_STEP:,}/{TARGET_STEPS:,} ({summary['progress_pct']}%), {left:,} steps left
--------------------------------------------------------------------""")
